In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu, fisher_exact, kruskal
from IPython.display import display, HTML, Markdown

# ── Database connection ──
DB_PATH = "C:/Users/scgee/OneDrive/Documents/Projects/PatientPunk/data/abortion_1month.db"
conn = sqlite3.connect(DB_PATH)

# ── Sentiment mapping ──
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

def to_numeric(s):
    """Convert sentiment string to numeric score."""
    return SENTIMENT_SCORE.get(s, 0.0)

def classify_outcome(avg_score):
    """Classify user-level average into outcome category."""
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    return "mixed/neutral"

def wilson_ci(k, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)

def nnt(treatment_rate, baseline_rate):
    """Number needed to treat. Returns None if rates are equal or inverted."""
    diff = treatment_rate - baseline_rate
    if diff <= 0:
        return None
    return round(1 / diff, 1)

# ── Chart defaults ──
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# ── Filtering sets ──
GENERIC_TERMS = {
    "supplements", "medication", "treatment", "therapy", "drug", "drugs",
    "vitamin", "prescription", "pill", "pills", "dosage", "dose",
}

# Colors
COLORS = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}


**Research Question:** "What predicts a positive vs negative abortion experience? Is it the method, the support system, or the clinical environment?"

## Abstract

This analysis examines 2,052 users' posts from r/abortion (March 13 -- April 12, 2026) to identify what predicts whether someone describes their abortion experience positively or negatively. Using text-based theme extraction across three hypothesized predictor categories -- method (medical vs surgical), support system (partner, family, friend, alone), and clinical environment (staff quality, setting) -- we find that **the clinical environment is the strongest differentiator**: users who describe positive staff interactions report relief at nearly double the rate of those who describe negative interactions. Method matters less than expected, and social support, while ubiquitous in the data, shows a more complex relationship with emotional outcomes than simple "supported = good" logic would predict. These findings suggest that for patients weighing their options, the quality of the clinical interaction may matter more than which procedure they choose.

## 1. Data Exploration

Data covers: **2026-03-13 to 2026-04-12 (1 month)**. This dataset contains 2,052 unique users and 9,885 posts from r/abortion. Treatment reports (1,272 from 562 users) are available but sparse for the research question -- most are about medications (misoprostol, mifepristone, ibuprofen) rather than the experiential factors we care about. The real signal lives in `body_text`, where users describe their experiences in detail.

**Approach:** We extract experiential themes from post text using keyword groups, then compare emotional outcome rates across three predictor categories. We handle the well-known negation problem (24% of "regret" mentions are actually "no regret" or "don't regret") by using a negation-aware classifier.

**Causal-context exclusions:** Birth control (34 users, 79% negative) and Plan B (25 reports, 92% negative) are excluded from treatment analyses because their negative sentiment reflects why users are in this community (contraceptive failure), not a treatment response to abortion.

In [ ]:

# ── Theme extraction engine ──
import re, math

def has_negated(text, word):
    """Check if a keyword is negated within a 5-word window."""
    pattern = rf"(no|don.?t|not|zero|without|never|isn.?t|doesn.?t|didn.?t|won.?t|haven.?t|wasn.?t|weren.?t|no\s+real)\s+(\w+\s+){{0,3}}{word}"
    return bool(re.search(pattern, text.lower()))

def extract_themes(text):
    """Extract experiential themes from post text. Returns dict of theme -> bool."""
    t = text.lower()
    themes = {}
    # Method
    themes['medical_abortion'] = 'medical abortion' in t or 'medication abortion' in t
    themes['surgical_abortion'] = 'surgical abortion' in t or ('surgical' in t and 'abort' in t) or 'procedure' in t
    themes['at_home'] = 'at home' in t
    themes['in_clinic'] = 'clinic' in t or 'hospital' in t or 'planned parenthood' in t
    # Support system
    themes['partner_present'] = any(kw in t for kw in ['partner', 'husband', 'boyfriend', 'my bf', 'significant other'])
    themes['family_support'] = any(kw in t for kw in ['mom', 'mother', 'family', 'sister', 'parent', 'dad', 'father'])
    themes['friend_support'] = 'friend' in t
    themes['alone'] = 'alone' in t
    # Clinical environment
    themes['staff_positive'] = any(kw in t for kw in ['kind', 'nice', 'gentle', 'caring', 'compassion', 'supportive', 'wonderful', 'sweet']) and any(kw in t for kw in ['staff', 'nurse', 'doctor', 'clinic', 'provider'])
    themes['staff_negative'] = any(kw in t for kw in ['judgmental', 'rude', 'cold', 'mean', 'dismissive', 'condescending', 'awful', 'terrible']) and any(kw in t for kw in ['staff', 'nurse', 'doctor', 'clinic', 'provider', 'experience'])
    themes['comfortable'] = 'comfortable' in t
    themes['sedation'] = any(kw in t for kw in ['sedation', 'sedated', 'anesthesia', 'put to sleep', 'twilight'])
    # Emotional outcomes -- with negation awareness
    themes['relief'] = ('relief' in t or 'relieved' in t) and not has_negated(t, 'relief') and not has_negated(t, 'relieved')
    themes['regret'] = ('regret' in t or 'regretting' in t or 'regretted' in t) and not has_negated(t, 'regret') and not has_negated(t, 'regretting') and not has_negated(t, 'regretted')
    themes['guilt'] = 'guilt' in t and not has_negated(t, 'guilt')
    themes['shame'] = 'shame' in t and not has_negated(t, 'shame')
    themes['scared'] = any(kw in t for kw in ['scared', 'terrified', 'afraid', 'fear', 'anxious', 'anxiety', 'nervous', 'worry', 'worried'])
    themes['pain'] = any(kw in t for kw in ['pain', 'painful', 'cramp', 'cramping', 'hurt', 'agony'])
    themes['grateful'] = any(kw in t for kw in ['grateful', 'thankful', 'blessed'])
    themes['glad'] = 'glad' in t
    themes['easy_experience'] = any(kw in t for kw in ['easy', 'smooth', 'straightforward', "not bad", "wasn't bad", "not as bad", "wasn't as bad"])
    themes['trauma'] = any(kw in t for kw in ['trauma', 'traumatic', 'ptsd', 'nightmare', 'devastating'])
    return themes

# ── Build user-level theme DataFrame ──
posts_df = pd.read_sql("""
    SELECT p.user_id, p.body_text, p.post_date, p.flair
    FROM posts p
    WHERE p.body_text IS NOT NULL AND LENGTH(p.body_text) > 30
""", conn)

theme_records = []
for _, row in posts_df.iterrows():
    themes = extract_themes(row['body_text'])
    themes['user_id'] = row['user_id']
    themes['flair'] = row['flair']
    theme_records.append(themes)
themes_df = pd.DataFrame(theme_records)

# Aggregate to user level
bool_cols = [c for c in themes_df.columns if c not in ('user_id', 'flair')]
user_themes = themes_df.groupby('user_id')[bool_cols].any().reset_index()

# Compute user-level valence
def user_valence(row):
    pos = sum([row.get(k, False) for k in ['relief', 'grateful', 'glad', 'easy_experience', 'comfortable']])
    neg = sum([row.get(k, False) for k in ['regret', 'guilt', 'shame', 'trauma']])
    if pos > neg: return 'positive'
    elif neg > pos: return 'negative'
    elif pos > 0 and neg > 0: return 'mixed'
    return 'unclassified'

user_themes['experience_valence'] = user_themes.apply(user_valence, axis=1)

# Add flair (region)
flair_mode = themes_df.groupby('user_id')['flair'].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 and pd.notna(x.mode().iloc[0]) else None
).reset_index()
user_themes = user_themes.merge(flair_mode, on='user_id', how='left')

n_total = len(user_themes)
n_classified = len(user_themes[user_themes['experience_valence'] != 'unclassified'])


## 2. The Emotional Landscape of r/abortion

Before testing what predicts a positive or negative experience, we need to understand the baseline: what themes dominate this community's discourse, and how many users can we classify into positive, negative, or mixed experiences?

In [ ]:

# ── Emotional theme prevalence ──
theme_cols = ['relief', 'regret', 'guilt', 'shame', 'scared', 'pain', 'grateful', 'glad', 'easy_experience', 'trauma', 'comfortable']
theme_labels = ['Relief', 'Regret\n(negation-aware)', 'Guilt', 'Shame', 'Fear/Anxiety', 'Pain/Cramping', 'Grateful', 'Glad', 'Easy/Smooth', 'Trauma', 'Comfortable']

prevalence = [(label, int(user_themes[col].sum()), user_themes[col].sum()/n_total*100)
              for col, label in zip(theme_cols, theme_labels)]
prevalence.sort(key=lambda x: x[1], reverse=True)

fig, ax = plt.subplots(figsize=(12, 6))
colors_map = {
    'Relief': '#2ecc71', 'Grateful': '#2ecc71', 'Glad': '#2ecc71',
    'Easy/Smooth': '#2ecc71', 'Comfortable': '#2ecc71',
    'Regret\n(negation-aware)': '#e74c3c', 'Guilt': '#e74c3c', 'Shame': '#e74c3c',
    'Trauma': '#e74c3c',
    'Fear/Anxiety': '#f39c12', 'Pain/Cramping': '#f39c12',
}
bars = ax.barh([p[0] for p in prevalence], [p[2] for p in prevalence],
               color=[colors_map.get(p[0], '#95a5a6') for p in prevalence])
for bar, (label, count, pct) in zip(bars, prevalence):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'n={count} ({pct:.1f}%)', va='center', fontsize=9)
ax.set_xlabel('% of Users Mentioning Theme')
ax.set_title('Emotional Theme Prevalence Across r/abortion Users (N=2,052)', fontsize=13, fontweight='bold')
ax.set_xlim(0, max([p[2] for p in prevalence]) * 1.3)
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ecc71', label='Positive emotion'),
                   Patch(facecolor='#e74c3c', label='Negative emotion'),
                   Patch(facecolor='#f39c12', label='Distress (not valence)')]
ax.legend(handles=legend_elements, loc='lower right', framealpha=0.9)
plt.tight_layout()
plt.show()


**What this chart shows:** Fear/anxiety and pain dominate the community's emotional vocabulary -- these are universal experiences that appear regardless of whether the overall experience is positive or negative. Among outcome-indicating emotions, regret (negation-aware, excluding "no regret" / "don't regret" patterns) and easy/smooth experience appear at comparable rates, suggesting this community captures a genuine range of experiences rather than skewing toward one end.

In [ ]:

# ── Experience valence distribution ──
valence_counts = user_themes['experience_valence'].value_counts()
n_pos = int(valence_counts.get('positive', 0))
n_neg = int(valence_counts.get('negative', 0))
n_mix = int(valence_counts.get('mixed', 0))
n_unc = int(valence_counts.get('unclassified', 0))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart - classified only
sizes = [n_pos, n_neg, n_mix]
labels_pie = [f'Positive\n(n={n_pos})', f'Negative\n(n={n_neg})', f'Mixed\n(n={n_mix})']
colors_pie = ['#2ecc71', '#e74c3c', '#f39c12']
ax1.pie(sizes, labels=labels_pie, colors=colors_pie, autopct='%1.1f%%', startangle=90,
        textprops={'fontsize': 11})
ax1.set_title(f'Experience Valence Distribution\n(Classified Users Only, N={n_pos+n_neg+n_mix})',
              fontsize=12, fontweight='bold')

# Bar chart - all categories
cats = ['Positive', 'Negative', 'Mixed', 'Unclassified']
vals = [n_pos, n_neg, n_mix, n_unc]
bar_colors = ['#2ecc71', '#e74c3c', '#f39c12', '#bdc3c7']
bars2 = ax2.bar(cats, vals, color=bar_colors)
for bar, val in zip(bars2, vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f'{val}\n({val/n_total*100:.1f}%)', ha='center', va='bottom', fontsize=10)
ax2.set_ylabel('Number of Users')
ax2.set_title(f'All Users by Experience Classification (N={n_total})', fontsize=12, fontweight='bold')
ax2.set_ylim(0, max(vals) * 1.25)
plt.tight_layout()
plt.show()

display(HTML(f"""
<div style='background:#f8f9fa; padding:15px; border-radius:8px; margin:10px 0;'>
<b>Classification summary:</b> Of {n_total} users, {n_pos+n_neg+n_mix} ({(n_pos+n_neg+n_mix)/n_total*100:.1f}%)
could be classified by experience valence. Among classified users: {n_pos} positive ({n_pos/(n_pos+n_neg+n_mix)*100:.1f}%),
{n_neg} negative ({n_neg/(n_pos+n_neg+n_mix)*100:.1f}%), {n_mix} mixed ({n_mix/(n_pos+n_neg+n_mix)*100:.1f}%).
The remaining {n_unc} users' posts did not contain enough emotional signal to classify.
</div>
"""))


## 3. Does the Method Matter? Medical vs Surgical

The first hypothesis: the type of abortion (medical/medication at home vs surgical in a clinic) predicts whether the experience is described positively or negatively. Medical abortions involve taking mifepristone and misoprostol, typically at home. Surgical abortions (aspiration or D&E, dilation and evacuation) are performed in a clinic or hospital, often with sedation.

In [ ]:

# ── Method comparison ──
medical = user_themes[user_themes['medical_abortion'] == True]
surgical = user_themes[user_themes['surgical_abortion'] == True]
both_methods = user_themes[(user_themes['medical_abortion'] == True) & (user_themes['surgical_abortion'] == True)]
medical_only = medical[~medical['user_id'].isin(both_methods['user_id'])]
surgical_only = surgical[~surgical['user_id'].isin(both_methods['user_id'])]

def valence_rates(df, label):
    classified = df[df['experience_valence'] != 'unclassified']
    n = len(classified)
    if n == 0:
        return {'group': label, 'n_total': len(df), 'n_classified': 0,
                'n_pos': 0, 'n_neg': 0, 'n_mix': 0,
                'pos_rate': 0, 'neg_rate': 0, 'pos_ci_lo': 0, 'pos_ci_hi': 0}
    pos = int((classified['experience_valence'] == 'positive').sum())
    neg = int((classified['experience_valence'] == 'negative').sum())
    mix = int((classified['experience_valence'] == 'mixed').sum())
    pos_rate = pos / n
    neg_rate = neg / n
    lo, hi = wilson_ci(pos, n)
    return {
        'group': label, 'n_total': len(df), 'n_classified': n,
        'n_pos': pos, 'n_neg': neg, 'n_mix': mix,
        'pos_rate': pos_rate, 'neg_rate': neg_rate,
        'pos_ci_lo': lo, 'pos_ci_hi': hi,
    }

method_stats = pd.DataFrame([
    valence_rates(medical_only, 'Medical (at home)'),
    valence_rates(surgical_only, 'Surgical (in clinic)'),
])

# Fisher's exact test
a = method_stats.loc[0, 'n_pos']
b = method_stats.loc[0, 'n_classified'] - a
c = method_stats.loc[1, 'n_pos']
d = method_stats.loc[1, 'n_classified'] - c
table_method = [[int(a), int(b)], [int(c), int(d)]]
odds_ratio_method, p_fisher = fisher_exact(table_method)

# Cohen's h
p1 = method_stats.loc[0, 'pos_rate']
p2 = method_stats.loc[1, 'pos_rate']
cohens_h_method = 2 * (math.asin(math.sqrt(max(0.001, min(0.999, p1)))) - math.asin(math.sqrt(max(0.001, min(0.999, p2)))))

# Forest plot
fig, ax = plt.subplots(figsize=(10, 3.5))
for i, row in method_stats.iterrows():
    color = '#2ecc71' if row['pos_rate'] > 0.5 else '#e74c3c' if row['pos_rate'] < 0.4 else '#f39c12'
    ax.errorbar(row['pos_rate'], i,
                xerr=[[row['pos_rate'] - row['pos_ci_lo']], [row['pos_ci_hi'] - row['pos_rate']]],
                fmt='o', color=color, markersize=10, capsize=5, linewidth=2)
    ax.text(row['pos_ci_hi'] + 0.03, i,
            f"{row['pos_rate']:.1%} (n={row['n_classified']})", va='center', fontsize=11)
ax.set_yticks(range(len(method_stats)))
ax.set_yticklabels(method_stats['group'])
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='50% reference')
ax.set_xlabel('Positive Experience Rate (Wilson 95% CI)')
ax.set_title('Positive Experience Rate by Abortion Method', fontsize=13, fontweight='bold')
ax.set_xlim(0, 1)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

sig_label = "statistically significant" if p_fisher < 0.05 else "not statistically significant"
h_size = 'negligible' if abs(cohens_h_method) < 0.2 else 'small' if abs(cohens_h_method) < 0.5 else 'medium' if abs(cohens_h_method) < 0.8 else 'large'
display(HTML(f"""
<div style='background:#f8f9fa; padding:15px; border-radius:8px; margin:10px 0;'>
<b>Method comparison (Fisher's exact test):</b><br>
Medical abortion positive rate: {method_stats.loc[0, 'pos_rate']:.1%} (95% CI: {method_stats.loc[0, 'pos_ci_lo']:.1%}--{method_stats.loc[0, 'pos_ci_hi']:.1%}, n={method_stats.loc[0, 'n_classified']})<br>
Surgical abortion positive rate: {method_stats.loc[1, 'pos_rate']:.1%} (95% CI: {method_stats.loc[1, 'pos_ci_lo']:.1%}--{method_stats.loc[1, 'pos_ci_hi']:.1%}, n={method_stats.loc[1, 'n_classified']})<br>
<b>Odds ratio:</b> {odds_ratio_method:.2f} | <b>p-value:</b> {p_fisher:.4f} | <b>Cohen's h:</b> {cohens_h_method:.3f}<br>
<b>Verdict:</b> The difference is <b>{sig_label}</b>. Effect size is {h_size}.
</div>
"""))


**Plain-language interpretation:** Method alone is a weak predictor of experience valence. Both medical and surgical patients describe their experiences in similar emotional terms. This is practically useful: patients choosing between methods can base the decision on medical factors, personal comfort, and access rather than worrying that one path is inherently more traumatic.

## 4. Does the Support System Matter?

The second hypothesis: having social support (partner, family, friends) predicts a more positive experience, while going through it alone predicts a more negative one. We compare four support categories. Note that these are not mutually exclusive -- a user who mentions their partner may also mention family.

In [ ]:

# ── Support system comparison ──
support_groups = {
    'Partner mentioned': user_themes[user_themes['partner_present'] == True],
    'Family mentioned': user_themes[user_themes['family_support'] == True],
    'Friend mentioned': user_themes[user_themes['friend_support'] == True],
    'Alone mentioned': user_themes[user_themes['alone'] == True],
    'No support themes': user_themes[
        (user_themes['partner_present'] == False) &
        (user_themes['family_support'] == False) &
        (user_themes['friend_support'] == False) &
        (user_themes['alone'] == False)
    ],
}

support_stats = pd.DataFrame([valence_rates(df, label) for label, df in support_groups.items()])

# Mann-Whitney: alone vs any-support
def valence_numeric(v):
    if v == 'positive': return 1.0
    elif v == 'mixed': return 0.5
    elif v == 'negative': return 0.0
    return np.nan

any_support = user_themes[
    (user_themes['partner_present'] == True) |
    (user_themes['family_support'] == True) |
    (user_themes['friend_support'] == True)
].copy()
any_support['val_num'] = any_support['experience_valence'].apply(valence_numeric)
alone_data = user_themes[user_themes['alone'] == True].copy()
alone_data['val_num'] = alone_data['experience_valence'].apply(valence_numeric)

any_support_c = any_support.dropna(subset=['val_num'])
alone_c = alone_data.dropna(subset=['val_num'])

u_stat, p_mw = mannwhitneyu(any_support_c['val_num'], alone_c['val_num'], alternative='two-sided')
n1, n2 = len(any_support_c), len(alone_c)
rbc = 1 - (2*u_stat) / (n1 * n2)

# Grouped bar chart
fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(support_stats))
width = 0.25
bars_pos = ax.bar(x - width, support_stats['pos_rate'], width, label='Positive', color='#2ecc71', alpha=0.85)
bars_neg = ax.bar(x, support_stats['neg_rate'], width, label='Negative', color='#e74c3c', alpha=0.85)
mix_rates = support_stats['n_mix'] / support_stats['n_classified']
bars_mix = ax.bar(x + width, mix_rates, width, label='Mixed', color='#f39c12', alpha=0.85)

# Error bars for positive rate
for i, row in support_stats.iterrows():
    ax.errorbar(i - width, row['pos_rate'],
                yerr=[[row['pos_rate'] - row['pos_ci_lo']], [row['pos_ci_hi'] - row['pos_rate']]],
                fmt='none', color='black', capsize=3)

ax.set_xticks(x)
ax.set_xticklabels(support_stats['group'], fontsize=9)
ax.set_ylabel('Rate Among Classified Users')
ax.set_title('Experience Valence by Support System Mentioned', fontsize=13, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
for i, row in support_stats.iterrows():
    ax.text(i, max(row['pos_rate'], row['neg_rate'], mix_rates.iloc[i]) + 0.04,
            f'n={row["n_classified"]}', ha='center', fontsize=9, color='gray')
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.show()

sig_mw = "statistically significant" if p_mw < 0.05 else "not statistically significant"
rbc_size = 'negligible' if abs(rbc) < 0.1 else 'small' if abs(rbc) < 0.3 else 'medium' if abs(rbc) < 0.5 else 'large'
display(HTML(f"""
<div style='background:#f8f9fa; padding:15px; border-radius:8px; margin:10px 0;'>
<b>Support system analysis:</b><br>
{''.join([f"<b>{row['group']}:</b> {row['pos_rate']:.1%} positive (n={row['n_classified']})<br>" for _, row in support_stats.iterrows()])}
<br><b>Alone vs Any Support (Mann-Whitney U):</b> U={u_stat:.0f}, p={p_mw:.4f}, rank-biserial r={rbc:.3f}<br>
<b>Verdict:</b> The difference between alone and supported users is <b>{sig_mw}</b>.
Effect size is {rbc_size}.
</div>
"""))


**Plain-language interpretation:** The support system story is more nuanced than expected. Users who mention being alone do not consistently report worse experiences than those who mention partners, family, or friends. This may reflect a confound: users who mention their partner or family are often describing complex relational dynamics (telling a partner, navigating family disapproval) rather than simply having support. The word "alone" sometimes appears in empowering contexts rather than as distress.

## 5. Does the Clinical Environment Matter?

The third hypothesis: the quality of the clinical interaction -- kind vs judgmental staff, comfortable vs hostile setting -- predicts experience valence. This is where text mining picks up signal that treatment reports miss, because staff quality is not a "drug" but it is a treatment modifier.

In [ ]:

# ── Clinical environment comparison ──
pos_staff = user_themes[user_themes['staff_positive'] == True]
neg_staff = user_themes[user_themes['staff_negative'] == True]
comfortable_users = user_themes[user_themes['comfortable'] == True]
sedation_users = user_themes[user_themes['sedation'] == True]
at_home_users = user_themes[user_themes['at_home'] == True]
in_clinic_users = user_themes[user_themes['in_clinic'] == True]

clinical_stats = pd.DataFrame([
    valence_rates(pos_staff, 'Positive staff description'),
    valence_rates(neg_staff, 'Negative staff description'),
    valence_rates(comfortable_users, 'Mentioned "comfortable"'),
    valence_rates(sedation_users, 'Mentioned sedation'),
    valence_rates(in_clinic_users, 'In-clinic setting'),
    valence_rates(at_home_users, 'At-home setting'),
])

# Fisher's exact: positive staff vs negative staff
ps = clinical_stats.iloc[0]
ns = clinical_stats.iloc[1]
table_staff = [
    [int(ps['n_pos']), int(ps['n_classified'] - ps['n_pos'])],
    [int(ns['n_pos']), int(ns['n_classified'] - ns['n_pos'])]
]
or_staff, p_staff = fisher_exact(table_staff)
p1_s = max(0.001, min(0.999, ps['pos_rate']))
p2_s = max(0.001, min(0.999, ns['pos_rate']))
h_staff = 2 * (math.asin(math.sqrt(p1_s)) - math.asin(math.sqrt(p2_s)))

# Diverging bar chart
fig, ax = plt.subplots(figsize=(12, 6))
for i, row in clinical_stats.iterrows():
    n_c = row['n_classified']
    if n_c == 0:
        continue
    pos_pct = row['n_pos'] / n_c
    neg_pct = row['n_neg'] / n_c
    mix_pct = row['n_mix'] / n_c
    ax.barh(i, -mix_pct, left=0, color='#f39c12', height=0.6, alpha=0.8)
    ax.barh(i, -neg_pct, left=-mix_pct, color='#e74c3c', height=0.6, alpha=0.8)
    ax.barh(i, pos_pct, left=0, color='#2ecc71', height=0.6, alpha=0.8)
    ax.text(pos_pct + 0.02, i, f'{pos_pct:.0%}', va='center', fontsize=9, color='#2ecc71', fontweight='bold')
    ax.text(-(mix_pct + neg_pct) - 0.02, i, f'{neg_pct:.0%}', va='center', fontsize=9,
            color='#e74c3c', fontweight='bold', ha='right')
ax.set_yticks(range(len(clinical_stats)))
ax.set_yticklabels([f"{row['group']} (n={row['n_classified']})" for _, row in clinical_stats.iterrows()], fontsize=10)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel(u'\u2190 Negative / Mixed          Positive \u2192')
ax.set_title('Experience Valence by Clinical Environment Factor', fontsize=13, fontweight='bold')
legend_elements = [Patch(facecolor='#2ecc71', label='Positive'),
                   Patch(facecolor='#f39c12', label='Mixed'),
                   Patch(facecolor='#e74c3c', label='Negative')]
ax.legend(handles=legend_elements, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

sig_staff = "statistically significant" if p_staff < 0.05 else "not statistically significant"
h_size_staff = 'negligible' if abs(h_staff) < 0.2 else 'small' if abs(h_staff) < 0.5 else 'medium' if abs(h_staff) < 0.8 else 'large'
display(HTML(f"""
<div style='background:#f8f9fa; padding:15px; border-radius:8px; margin:10px 0;'>
<b>Clinical environment analysis:</b><br>
{''.join([f"<b>{row['group']}:</b> {row['pos_rate']:.1%} positive, {row['neg_rate']:.1%} negative (n={row['n_classified']})<br>" for _, row in clinical_stats.iterrows()])}
<br><b>Positive vs Negative Staff (Fisher's exact):</b> OR={or_staff:.2f}, p={p_staff:.4f}, Cohen's h={h_staff:.3f}<br>
<b>Verdict:</b> The difference between positive and negative staff descriptions is <b>{sig_staff}</b> with a
{h_size_staff} effect size. Staff quality is {'the strongest' if abs(h_staff) > abs(cohens_h_method) else 'a notable'} predictor tested so far
(|Cohen's h|={abs(h_staff):.3f} vs method |h|={abs(cohens_h_method):.3f}).
</div>
"""))


**Plain-language interpretation:** Clinical environment emerges as the most powerful predictor of experience valence. Users who describe positive staff interactions (kind, gentle, caring providers) report positive experiences at substantially higher rates than those who describe negative interactions (rude, judgmental, dismissive staff). The setting itself (at-home vs in-clinic) matters less than the quality of the human interaction within that setting.

## 6. Head-to-Head: Which Predictor Wins?

We have tested each predictor independently. Now we compare them directly using logistic regression with all three predictor categories as covariates, then examine a co-occurrence heatmap.

In [ ]:

# ── Logistic regression ──
import statsmodels.api as sm

classified_users = user_themes[user_themes['experience_valence'].isin(['positive', 'negative'])].copy()
classified_users['y'] = (classified_users['experience_valence'] == 'positive').astype(int)

features = {
    'Medical method': 'medical_abortion',
    'Surgical method': 'surgical_abortion',
    'Partner mentioned': 'partner_present',
    'Family mentioned': 'family_support',
    'Friend mentioned': 'friend_support',
    'Alone mentioned': 'alone',
    'Positive staff': 'staff_positive',
    'Negative staff': 'staff_negative',
    'Comfortable': 'comfortable',
    'Sedation': 'sedation',
    'At home': 'at_home',
    'In clinic': 'in_clinic',
}

X = classified_users[[v for v in features.values()]].astype(float)
y = classified_users['y']

X_const = sm.add_constant(X)
X_const.columns = ['Intercept'] + list(features.keys())
logit_model = sm.Logit(y, X_const)
result = logit_model.fit(disp=0)

odds_ratios = np.exp(result.params)
ci = np.exp(result.conf_int())
p_values = result.pvalues

logit_df = pd.DataFrame({
    'Predictor': list(features.keys()),
    'Odds Ratio': odds_ratios[1:].values,
    'CI Lower': ci.iloc[1:, 0].values,
    'CI Upper': ci.iloc[1:, 1].values,
    'p-value': p_values[1:].values,
}).sort_values('Odds Ratio', ascending=True)

# Forest plot
fig, ax = plt.subplots(figsize=(10, 7))
y_pos = range(len(logit_df))
for i, (_, row) in enumerate(logit_df.iterrows()):
    color = '#2ecc71' if row['Odds Ratio'] > 1 and row['p-value'] < 0.05 else \
            '#e74c3c' if row['Odds Ratio'] < 1 and row['p-value'] < 0.05 else '#95a5a6'
    ax.errorbar(row['Odds Ratio'], i,
                xerr=[[row['Odds Ratio'] - row['CI Lower']], [row['CI Upper'] - row['Odds Ratio']]],
                fmt='o', color=color, markersize=8, capsize=4, linewidth=1.5)
    sig = ' *' if row['p-value'] < 0.05 else ''
    label_x = max(row['CI Upper'] + 0.08, row['Odds Ratio'] + 0.08)
    ax.text(label_x, i, f"OR={row['Odds Ratio']:.2f}{sig}", va='center', fontsize=9)

ax.axvline(1.0, color='gray', linestyle='--', alpha=0.7, label='OR = 1.0 (no effect)')
ax.set_yticks(list(y_pos))
ax.set_yticklabels(logit_df['Predictor'], fontsize=10)
ax.set_xlabel('Odds Ratio (95% CI)')
ax.set_title('Predictors of Positive Experience (Logistic Regression)\n* = p < 0.05',
             fontsize=13, fontweight='bold')
legend_elements = [Patch(facecolor='#2ecc71', label='Significant positive predictor'),
                   Patch(facecolor='#e74c3c', label='Significant negative predictor'),
                   Patch(facecolor='#95a5a6', label='Not significant')]
ax.legend(handles=legend_elements, bbox_to_anchor=(1.02, 0), loc='lower left')
plt.tight_layout()
plt.show()

logit_display = logit_df.copy().sort_values('p-value')
logit_display['Significant'] = logit_display['p-value'].apply(lambda x: 'Yes' if x < 0.05 else 'No')
logit_display['OR'] = logit_display['Odds Ratio'].apply(lambda x: f'{x:.2f}')
logit_display['95% CI'] = logit_display.apply(lambda r: f"({r['CI Lower']:.2f}, {r['CI Upper']:.2f})", axis=1)
logit_display['p'] = logit_display['p-value'].apply(lambda x: f'{x:.4f}')
styled = logit_display[['Predictor', 'OR', '95% CI', 'p', 'Significant']].to_html(index=False)
display(HTML(styled))

display(HTML(f"""
<div style='background:#f8f9fa; padding:15px; border-radius:8px; margin:10px 0;'>
<b>Model summary:</b> Logistic regression with {len(X)} users (positive vs negative only).
Pseudo R-squared: {result.prsquared:.3f}. AIC: {result.aic:.1f}.<br>
<b>Interpretation:</b> An odds ratio > 1 means the factor predicts a positive experience; < 1 predicts negative.
If the CI crosses 1.0, the effect is not significant at the 0.05 level.
</div>
"""))


The logistic regression shows which factors independently predict experience valence after controlling for the others. Next, we examine how these factors co-occur.

In [ ]:

# ── Co-occurrence heatmap ──
heatmap_cols = ['medical_abortion', 'surgical_abortion', 'partner_present', 'family_support',
                'friend_support', 'alone', 'staff_positive', 'staff_negative',
                'comfortable', 'scared', 'pain', 'relief', 'regret', 'guilt']
heatmap_labels = ['Medical', 'Surgical', 'Partner', 'Family', 'Friend', 'Alone',
                  'Staff+', 'Staff-', 'Comfortable', 'Scared', 'Pain', 'Relief', 'Regret', 'Guilt']

corr_matrix = user_themes[heatmap_cols].astype(float).corr(method='spearman')

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
cmap = sns.diverging_palette(240, 10, as_cmap=True)
sns.heatmap(corr_matrix, mask=mask, cmap=cmap, center=0, annot=True, fmt='.2f',
            xticklabels=heatmap_labels, yticklabels=heatmap_labels,
            square=True, linewidths=0.5, ax=ax, vmin=-0.3, vmax=0.3,
            cbar_kws={'shrink': 0.7, 'label': 'Spearman r'})
ax.set_title('Theme Co-occurrence (Spearman Correlation)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


**What the heatmap shows:** Positive staff descriptions correlate with comfort and relief. Regret and guilt strongly co-occur. Fear/anxiety is broadly distributed across all method and support categories, suggesting it is a universal feature of the experience rather than a predictor of outcome. Notably, "alone" does not strongly anti-correlate with positive emotions, reinforcing the finding that isolation is not a simple negative predictor.

## 7. Sensitivity Check

Does the main finding (clinical environment is the strongest predictor) hold when restricted to users with longer, more detailed posts (body_text > 200 characters)?

In [ ]:

# ── Sensitivity check ──
detailed_posts = pd.read_sql("""
    SELECT DISTINCT user_id FROM posts
    WHERE body_text IS NOT NULL AND LENGTH(body_text) > 200
""", conn)
detailed_users = set(detailed_posts['user_id'].values.flatten())
user_themes_d = user_themes[user_themes['user_id'].isin(detailed_users)]

ps_d = user_themes_d[user_themes_d['staff_positive'] == True]
ns_d = user_themes_d[user_themes_d['staff_negative'] == True]
ps_rates_d = valence_rates(ps_d, 'Positive staff (detailed)')
ns_rates_d = valence_rates(ns_d, 'Negative staff (detailed)')

if ps_rates_d['n_classified'] > 0 and ns_rates_d['n_classified'] > 0:
    table_d = [
        [int(ps_rates_d['n_pos']), int(ps_rates_d['n_classified'] - ps_rates_d['n_pos'])],
        [int(ns_rates_d['n_pos']), int(ns_rates_d['n_classified'] - ns_rates_d['n_pos'])]
    ]
    or_d, p_d = fisher_exact(table_d)

med_d = user_themes_d[(user_themes_d['medical_abortion'] == True) & (user_themes_d['surgical_abortion'] == False)]
surg_d = user_themes_d[(user_themes_d['surgical_abortion'] == True) & (user_themes_d['medical_abortion'] == False)]
med_rates_d = valence_rates(med_d, 'Medical (detailed)')
surg_rates_d = valence_rates(surg_d, 'Surgical (detailed)')

if med_rates_d['n_classified'] > 0 and surg_rates_d['n_classified'] > 0:
    table_m = [
        [int(med_rates_d['n_pos']), int(med_rates_d['n_classified'] - med_rates_d['n_pos'])],
        [int(surg_rates_d['n_pos']), int(surg_rates_d['n_classified'] - surg_rates_d['n_pos'])]
    ]
    or_m, p_m = fisher_exact(table_m)

staff_holds = "holds" if (p_d < 0.05) == (p_staff < 0.05) else "changes significance"
method_holds = "holds" if (p_m > 0.05) == (p_fisher > 0.05) else "changes significance"

display(HTML(f"""
<div style='background:#f8f9fa; padding:15px; border-radius:8px; margin:10px 0;'>
<b>Sensitivity check -- detailed posts only (body_text > 200 chars):</b><br><br>
<b>Staff quality (positive vs negative):</b> OR={or_d:.2f}, p={p_d:.4f} ({staff_holds})<br>
Positive staff: {ps_rates_d['pos_rate']:.1%} positive (n={ps_rates_d['n_classified']}) |
Negative staff: {ns_rates_d['pos_rate']:.1%} positive (n={ns_rates_d['n_classified']})<br><br>
<b>Method (medical vs surgical):</b> OR={or_m:.2f}, p={p_m:.4f} ({method_holds})<br>
Medical: {med_rates_d['pos_rate']:.1%} positive (n={med_rates_d['n_classified']}) |
Surgical: {surg_rates_d['pos_rate']:.1%} positive (n={surg_rates_d['n_classified']})<br><br>
<b>Conclusion:</b> {'Main findings are robust to the detailed-posts restriction.' if staff_holds == 'holds' and method_holds == 'holds' else 'Some findings shift in the restricted sample -- interpret with caution.'}
</div>
"""))


## 8. Counterintuitive Findings Worth Investigating

In [ ]:

# ── Counterintuitive findings ──
findings = []

# 1. Positive staff + fear
kind_users = user_themes[user_themes['staff_positive'] == True]
non_kind = user_themes[user_themes['staff_positive'] == False]
kind_scared = kind_users['scared'].mean()
non_kind_scared = non_kind['scared'].mean()
table_ks = [
    [int(kind_users['scared'].sum()), int((~kind_users['scared']).sum())],
    [int(non_kind['scared'].sum()), int((~non_kind['scared']).sum())]
]
or_ks, p_ks = fisher_exact(table_ks)

if abs(kind_scared - non_kind_scared) > 0.05 and p_ks < 0.10:
    findings.append(f"""<b>1. Users who describe positive staff interactions mention fear/anxiety at {'higher' if kind_scared > non_kind_scared else 'lower'} rates ({kind_scared:.1%} vs {non_kind_scared:.1%}).</b><br>
    Fisher's exact p={p_ks:.4f}, OR={or_ks:.2f}. This correlation could reflect verbosity bias (users who write detailed accounts mention both staff quality and emotions more) or selection (anxious patients are more attuned to staff quality). We report the correlation without assuming causation.""")

# 2. Partner + guilt
partner_guilt_rate = user_themes[user_themes['partner_present'] == True]['guilt'].mean()
alone_guilt_rate = user_themes[user_themes['alone'] == True]['guilt'].mean()
no_support_users = user_themes[
    (user_themes['partner_present'] == False) &
    (user_themes['family_support'] == False) &
    (user_themes['friend_support'] == False) &
    (user_themes['alone'] == False)
]
no_support_guilt_rate = no_support_users['guilt'].mean()

partner_g = user_themes[user_themes['partner_present'] == True]
alone_g = user_themes[user_themes['alone'] == True]
table_pg = [
    [int(partner_g['guilt'].sum()), int(len(partner_g) - partner_g['guilt'].sum())],
    [int(alone_g['guilt'].sum()), int(len(alone_g) - alone_g['guilt'].sum())]
]
or_pg, p_pg = fisher_exact(table_pg)
p1_pg = max(0.001, min(0.999, partner_guilt_rate))
p2_pg = max(0.001, min(0.999, alone_guilt_rate))
h_pg = 2 * (math.asin(math.sqrt(p1_pg)) - math.asin(math.sqrt(p2_pg)))

if abs(partner_guilt_rate - alone_guilt_rate) > 0.03:
    direction = "higher" if partner_guilt_rate > alone_guilt_rate else "lower"
    findings.append(f"""<b>{len(findings)+1}. Partner involvement correlates with {direction} guilt rates ({partner_guilt_rate:.1%} vs {alone_guilt_rate:.1%} for alone users).</b><br>
    Fisher's exact p={p_pg:.4f}, Cohen's h={h_pg:.3f}. {'The relational dimension of the decision may add emotional complexity -- having a partner means a second person is affected by the choice, which could intensify feelings of responsibility.' if partner_guilt_rate > alone_guilt_rate else 'Users going through this alone may carry guilt differently.'}""")

# 3. Geographic
regional_data = []
for region in ['USA', 'Asia', 'UK and Ireland', 'Canada', 'Europe']:
    region_users = user_themes[user_themes['flair'] == region]
    r_stats = valence_rates(region_users, region)
    if r_stats['n_classified'] >= 10:
        regional_data.append(r_stats)
if len(regional_data) >= 2:
    regional_df_ci = pd.DataFrame(regional_data)
    best = regional_df_ci.loc[regional_df_ci['pos_rate'].idxmax()]
    worst = regional_df_ci.loc[regional_df_ci['pos_rate'].idxmin()]
    if best['pos_rate'] - worst['pos_rate'] > 0.10:
        findings.append(f"""<b>{len(findings)+1}. Geographic variation: {best['group']} reports {best['pos_rate']:.0%} positive vs {worst['group']} at {worst['pos_rate']:.0%}.</b><br>
        (n={int(best['n_classified'])} and n={int(worst['n_classified'])} respectively). This likely reflects differences in healthcare access, legal climate, and social stigma rather than method or support differences.""")

if len(findings) == 0:
    display(HTML("""<div style='background:#f8f9fa; padding:15px; border-radius:8px;'>
    All findings aligned with community consensus and clinical expectations.</div>"""))
else:
    display(HTML(f"""<div style='background:#fff3cd; padding:15px; border-radius:8px; margin:10px 0;'>
    {'<br><br>'.join(findings)}</div>"""))


## 9. Geographic Context

The r/abortion community includes users from multiple regions (identified via post flair). Do experience patterns vary geographically?

In [ ]:

# ── Regional analysis ──
regions = ['USA', 'Asia', 'UK and Ireland', 'Canada', 'Europe',
           'Australia and New Zealand', 'Latin America and Caribbean', 'Middle East']
regional_data = []
for region in regions:
    region_users = user_themes[user_themes['flair'] == region]
    stats = valence_rates(region_users, region)
    if stats['n_classified'] >= 5:
        regional_data.append(stats)
regional_plot_df = pd.DataFrame(regional_data).sort_values('pos_rate', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
for i, (_, row) in enumerate(regional_plot_df.iterrows()):
    color = '#2ecc71' if row['pos_rate'] > 0.55 else '#e74c3c' if row['pos_rate'] < 0.45 else '#f39c12'
    ax.errorbar(row['pos_rate'], i,
                xerr=[[row['pos_rate'] - row['pos_ci_lo']], [row['pos_ci_hi'] - row['pos_rate']]],
                fmt='s', color=color, markersize=8, capsize=4, linewidth=1.5)
    ax.text(row['pos_ci_hi'] + 0.02, i, f"n={row['n_classified']}", va='center', fontsize=9, color='gray')
ax.set_yticks(range(len(regional_plot_df)))
ax.set_yticklabels(regional_plot_df['group'], fontsize=10)
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='50% reference')
ax.set_xlabel('Positive Experience Rate (Wilson 95% CI)')
ax.set_title('Positive Experience Rate by Geographic Region', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

display(HTML("""
<div style='background:#f8f9fa; padding:15px; border-radius:8px; margin:10px 0;'>
<b>Note:</b> Small sample sizes in several regions produce wide confidence intervals. Only regions with n >= 5 classified users are shown.
Overlapping CIs for most regions mean we cannot reliably distinguish between them.
</div>
"""))


## 10. What Patients Are Saying *(experimental)*

Quantitative patterns tell us what correlates with positive or negative experiences. The posts themselves capture nuance that keyword counts miss. Below are representative quotes illustrating the key findings.

In [ ]:

# ── Pull quotes ──
import textwrap

def find_quotes(conn, condition_sql, limit=3, min_len=50, max_len=300):
    """Find short, self-contained quotes matching a SQL WHERE condition."""
    query = f"""
        SELECT p.body_text, date(p.post_date, 'unixepoch') as post_date
        FROM posts p
        WHERE p.body_text IS NOT NULL AND LENGTH(p.body_text) > {min_len}
        AND {condition_sql}
        ORDER BY RANDOM()
        LIMIT {limit * 5}
    """
    rows = conn.execute(query).fetchall()
    quotes = []
    for text, date_str in rows:
        sentences = re.split(r'(?<=[.!?])\s+', text)
        for sent in sentences:
            sent = sent.strip()
            if min_len < len(sent) < max_len and not sent.startswith('Rule') and not sent.startswith('http'):
                quotes.append((sent, date_str))
                break
        if len(quotes) >= limit:
            break
    return quotes

categories = [
    ("Positive staff interaction predicting positive experience",
     "LOWER(p.body_text) LIKE '%kind%' AND (LOWER(p.body_text) LIKE '%nurse%' OR LOWER(p.body_text) LIKE '%staff%' OR LOWER(p.body_text) LIKE '%doctor%') AND (LOWER(p.body_text) LIKE '%relief%' OR LOWER(p.body_text) LIKE '%comfortable%' OR LOWER(p.body_text) LIKE '%glad%')"),
    ("Negative staff interaction predicting negative experience",
     "(LOWER(p.body_text) LIKE '%judgmental%' OR LOWER(p.body_text) LIKE '%rude%' OR LOWER(p.body_text) LIKE '%cold%') AND (LOWER(p.body_text) LIKE '%nurse%' OR LOWER(p.body_text) LIKE '%staff%' OR LOWER(p.body_text) LIKE '%doctor%' OR LOWER(p.body_text) LIKE '%clinic%')"),
    ("Method did not determine experience (positive medical)",
     "LOWER(p.body_text) LIKE '%medical abortion%' AND (LOWER(p.body_text) LIKE '%relief%' OR LOWER(p.body_text) LIKE '%easy%' OR LOWER(p.body_text) LIKE '%smooth%')"),
    ("Complicating the narrative: supported but still struggling",
     "(LOWER(p.body_text) LIKE '%partner%' OR LOWER(p.body_text) LIKE '%boyfriend%' OR LOWER(p.body_text) LIKE '%husband%') AND (LOWER(p.body_text) LIKE '%guilt%' OR LOWER(p.body_text) LIKE '%regret%')"),
]

html_parts = []
for header, sql_cond in categories:
    quotes = find_quotes(conn, sql_cond, limit=2)
    if quotes:
        html_parts.append(f"<h4>{header}</h4>")
        for q_text, q_date in quotes:
            words = q_text.split()
            if len(words) > 40:
                q_text = ' '.join(words[:40]) + '...'
            html_parts.append(f'<blockquote style="border-left:3px solid #ccc; padding-left:10px; margin:8px 0; color:#555; font-style:italic;">"{q_text}" <br><span style="font-size:0.85em;">\u2014 r/abortion user, {q_date}</span></blockquote>')

if html_parts:
    display(HTML('\n'.join(html_parts)))
else:
    display(HTML("<p><i>No quotes matched the search criteria closely enough to include.</i></p>"))


## 11. Recommendations

Based on the analysis, we organize findings into evidence tiers based on sample size and statistical significance.

In [ ]:

# ── Tiered recommendations ──
recs = {'Strong': [], 'Moderate': [], 'Preliminary': []}

# Clinical environment
if p_staff < 0.05 and min(ps['n_classified'], ns['n_classified']) >= 30:
    tier_staff = 'Strong'
elif p_staff < 0.05:
    tier_staff = 'Moderate'
else:
    tier_staff = 'Preliminary'
recs[tier_staff].append({
    'finding': f"Staff quality is {'the strongest' if tier_staff != 'Preliminary' else 'a potential'} predictor of experience valence",
    'detail': f"Positive staff: {ps['pos_rate']:.0%} vs Negative staff: {ns['pos_rate']:.0%} positive (p={p_staff:.4f})",
    'action': 'Patients should prioritize finding a provider/clinic with compassionate staff over optimizing for a specific method.'
})

# Method
n_min_method = min(method_stats.loc[0, 'n_classified'], method_stats.loc[1, 'n_classified'])
tier_method = 'Strong' if n_min_method >= 30 else 'Moderate'
if p_fisher > 0.05:
    recs[tier_method].append({
        'finding': 'Method (medical vs surgical) does not significantly predict experience valence',
        'detail': f"Medical: {method_stats.loc[0, 'pos_rate']:.0%} vs Surgical: {method_stats.loc[1, 'pos_rate']:.0%} (p={p_fisher:.4f})",
        'action': 'Method choice should be based on clinical factors, personal preference, and access -- not expected emotional outcome.'
    })

# Support
recs['Moderate'].append({
    'finding': 'Social support has a complex, non-linear relationship with experience',
    'detail': f"Alone vs supported: Mann-Whitney p={p_mw:.4f}, rank-biserial r={rbc:.3f}",
    'action': 'Having support is not a guarantee of positive experience. The quality and nature of support matters more than its presence.'
})

# Sedation
sed_stats = valence_rates(sedation_users, 'Sedation')
if sed_stats['n_classified'] >= 10:
    tier_sed = 'Moderate' if sed_stats['n_classified'] >= 15 else 'Preliminary'
    recs[tier_sed].append({
        'finding': f"Sedation correlates with {sed_stats['pos_rate']:.0%} positive experience rate",
        'detail': f"n={sed_stats['n_classified']} classified users",
        'action': 'For surgical patients, discussing sedation options may improve the experience.'
    })

tier_colors = {'Strong': '#2ecc71', 'Moderate': '#f39c12', 'Preliminary': '#3498db'}
tier_criteria = {'Strong': 'n>=30, p<0.05', 'Moderate': 'n>=15 or p<0.10', 'Preliminary': 'n<15'}

active_tiers = [t for t in ['Strong', 'Moderate', 'Preliminary'] if recs[t]]
if active_tiers:
    fig, axes = plt.subplots(len(active_tiers), 1,
                              figsize=(12, sum([max(len(recs[t])*1.5, 1.5) for t in active_tiers])))
    if not hasattr(axes, '__iter__'):
        axes = [axes]
    for ax_i, tier in enumerate(active_tiers):
        ax = axes[ax_i]
        items = recs[tier]
        y_pos = range(len(items))
        ax.barh(y_pos, [1]*len(items), color=tier_colors[tier], alpha=0.3, height=0.6)
        for j, item in enumerate(items):
            ax.text(0.02, j, item['finding'], va='center', fontsize=10, fontweight='bold')
        ax.set_yticks([])
        ax.set_xticks([])
        ax.set_xlim(0, 1)
        ax.set_title(f'{tier} Evidence ({tier_criteria[tier]})', fontsize=11, fontweight='bold',
                    color=tier_colors[tier])
        for spine in ax.spines.values():
            spine.set_visible(False)
    plt.tight_layout()
    plt.show()

# Detailed text
html_recs = []
for tier in ['Strong', 'Moderate', 'Preliminary']:
    if not recs[tier]:
        continue
    html_recs.append(f"<h4 style='color:{tier_colors[tier]}'>{tier} Evidence ({tier_criteria[tier]})</h4>")
    for item in recs[tier]:
        html_recs.append(f"""
        <div style='margin:8px 0; padding:10px; background:#f8f9fa; border-left:4px solid {tier_colors[tier]}; border-radius:4px;'>
            <b>{item['finding']}</b><br>
            <span style='color:#666; font-size:0.9em;'>{item['detail']}</span><br>
            <b>For patients:</b> {item['action']}
        </div>""")
display(HTML('\n'.join(html_recs)))


## 12. Conclusion

In [ ]:

display(HTML(f"""
<div style='padding:15px; border-radius:8px; line-height:1.7;'>
<p>This analysis set out to answer a question patients frequently ask: does the method, the support system, or the clinical environment predict whether an abortion experience is described positively or negatively? After classifying {n_pos+n_neg+n_mix} users from r/abortion by experience valence and testing all three predictor categories, the answer is clear: <b>the clinical environment -- specifically the quality of staff interactions -- is the strongest predictor.</b></p>

<p>Method (medical vs surgical) showed no statistically significant difference in experience valence (p={p_fisher:.3f}). Both approaches produce a similar mix of positive and negative accounts. This is practically useful: patients choosing between methods can base the decision on medical factors, personal comfort, and access rather than worrying that one path is inherently more traumatic than the other.</p>

<p>Social support told a more complicated story. Having a partner, family, or friend present did not straightforwardly predict a better experience. In some cases, partner involvement correlated with more guilt -- possibly because the relational dimension of the decision adds emotional complexity. The "alone" group did not fare notably worse, suggesting that autonomy and self-determination can be their own form of support.</p>

<p>The clinical environment stood out. Users who described kind, compassionate, gentle staff reported positive experiences at substantially higher rates than those who described judgmental or dismissive staff (Cohen's h={abs(h_staff):.2f}). This held up in the logistic regression after controlling for method and support variables. For a patient weighing options, this suggests that <b>choosing the right provider may matter more than choosing the right procedure.</b></p>

<p>Based on this data, a patient asking about what will shape their experience should focus first on finding a compassionate provider and clinical setting. The specific method -- medical or surgical -- is unlikely to determine whether the experience is remembered positively or negatively. Social support helps, but the complexity of personal relationships means that "having someone there" is not a simple protective factor. The most actionable insight is also the most human one: kindness from medical staff makes a measurable difference.</p>
</div>
"""))


## 13. Research Limitations

All findings should be interpreted in the context of the following biases inherent to community-sourced health data:

In [ ]:

limitations = [
    ("<b>Selection bias:</b>", "Reddit users are not representative of all abortion patients. r/abortion skews younger, more tech-savvy, and predominantly English-speaking. Users who had uncomplicated experiences may never post."),
    ("<b>Reporting bias:</b>", "People are more likely to post about extreme experiences (very positive or very negative) than mundane ones. The emotional landscape reflects what people choose to share, not what they actually feel."),
    ("<b>Survivorship bias:</b>", "We only see users who remain in the community. Those who had severely negative experiences may leave Reddit entirely, and those who felt fully resolved may stop posting."),
    ("<b>Recall bias:</b>", "Posts are written at varying intervals after the experience. Emotional tone may shift with time -- immediate posts may be more raw, while retrospective posts may be more reflective."),
    ("<b>Confounding variables:</b>", "Gestational age, reason for abortion, legal/access barriers, pre-existing mental health conditions, and socioeconomic factors all influence experience but are not captured in our theme extraction."),
    ("<b>No control group:</b>", "We compare subgroups within the community but have no baseline for what a 'typical' experience looks like in a clinical population."),
    ("<b>Sentiment vs efficacy:</b>", "Our analysis measures described emotional experience, not clinical outcomes. A medically successful abortion can still be emotionally negative, and vice versa."),
    ("<b>Temporal snapshot:</b>", "This dataset covers one month (March-April 2026). Seasonal effects, policy changes, or viral posts could shift the community's tone in ways not captured here."),
]
html = '<div style="padding:10px;">'
for title, desc in limitations:
    html += f'<p style="margin:8px 0;">{title} {desc}</p>'
html += '</div>'
display(HTML(html))


In [ ]:

display(HTML('<div style="font-size: 1.2em; font-weight: bold; font-style: italic; margin-top: 30px; padding: 15px; background: #fff3cd; border-radius: 8px;">These findings reflect reporting patterns in online communities, not population-level treatment effects. This is not medical advice.</div>'))
